# Древовидные структуры данных

In [ ]:
import re
import random

random.seed(42)

## Самостоятельная работа

Вопросы:
- Назовите один алгоритм сортировки и его вычислительную сложность, кратко опишите алгоритм работы (можно словами, можно кодом на питоне / псевдокодом);
- У вас есть неотсортированный массив из $N$ элементов ($N > 2$). Нужно проверять наличие в нем элементов. Вы можете либо каждый раз делать линейный поиск, либо один раз отсортировать массив (эффективным методом) и затем использовать бинарный поиск.

    Что будет эффективнее, если поиск выполняется один раз? А если поиск выполняется $k$ раз для разных значений? Обоснуйте ответ через вычислительную сложность и оцените, при каких $k$ выгодно сначала отсортировать массив.

## Префиксные/суффиксные деревья

Если у нас есть некоторый словарь, который хранит слова упорядоченными по алфавиту, то из всего словаря можно выделить подмножество слов, начинающихся на одну букву. Если словарь велик, возникает вопрос: зачем хранить эту букву многократно для каждого слова? В этом случае можно сгруппировать данные: создать структуру, в которой ключом будет первая буква слова, а значением — список всех слов, начинающихся на неё.

Однако и такие списки могут быть очень большими. Тогда естественно применить ту же идею ещё раз: разбивать слова не только по первой букве, но и по последующим позициям. В результате получается иерархическая структура, в которой каждый уровень отвечает за выбор следующего символа, а переходы зависят от уже накопленного префикса. Таким образом, мы фактически приходим к структуре, где слова хранятся не целиком, а собираются по символам при движении вниз по дереву — то есть к префиксному дереву (trie).

<img src="https://github.com/klyshinsky/Math4Linguists2025/blob/main/img/morfologiya-zadachi-i-podhody-k-ih-resheniyu-4.png?raw=1">

До этого мы говорили про алгоритмы, позволяющие делать эффективную сортировку данных, однако сами данные мы всегда полагали плоскими (массив, список). При этом, часто бывает выгодно отойти от такой структуры в пользу более сложной, чтобы одновременно ускорить поиск и более эффективно использовать память, используя особенности наших данных.

Для текстов (и любых последовательностей) хорошим примером такой структуры является префиксное или суффиксное дерево.

В чем идея: будем раскладывать наши данные в дерево так, чтобы один элемент собирался одним прохождением от корня вниз по дереву. Как будет выглядеть поиск?

Алгоритм поиска:
1. В качестве текущей вершины берем корень префиксного дерева, а в качестве текущей буквы - первую букву слова.
2. Ищем текущую букву среди потомков текущей вершины.
    - Если нашли, переходим на уровень ниже. В качестве текущей вершины назначаем вершину, ассоциированную с текущей буквой. В качестве текущей назначаем следующую букву.
    - Если не нашли, завершаем алгоритм с сообщением, что такого слова нет в словаре.
3. Повторяем шаг 2, пока слово не закончится (при попытке взять следующую букву ее не оказалось).  

Этот алгоритм будет иметь вычислительную сложность $O(l)$, где $l$ - это длина текущего слова, которое надо найти.

В хороших реализациях алгоритма еще разделяют буквы, которые могут быть окончанием слова, и буквы, которые не могут. Например, считают конец строки отдельным символом, который должен найтись среди потомков последней буквы, иначе такого слова не существует в словаре.

Вычислительная сложность построения такого дерева менее очевидна, чем у поиска, так как она зависит от длины слов:

$$
O\!\left(\textstyle \sum \text{длин всех слов} \right) = O(n \cdot L),
$$
где $L$ - средняя длина слова в словаре.

Давайте попробуем построить дерево на примере короткого текста про котят:

> Утром у подъезда появился кот. Сначала его заметил маленький котик, который жил в соседнем дворе. Котик подошёл ближе и увидел, что это не просто кот, а пушистый котёнок, который потерялся. Вскоре вокруг собрались другие котята, живущие неподалёку. Все котята начали мяукать и звать взрослых, чтобы помочь найти коту дом.



In [ ]:
text = 'Утром у подъезда появился кот. Сначала его заметил маленький котик, который жил в соседнем дворе. Котик подошёл ближе и увидел, ' \
    'что это не просто кот, а пушистый котёнок, который потерялся. Вскоре вокруг собрались другие котята, живущие неподалёку. Все котята ' \
    'начали мяукать и звать взрослых, чтобы помочь найти коту дом.'
words = list(map(str.lower, re.findall("[А-ЯЁа-яё]+", text)))
words[:10]

['утром',
 'у',
 'подъезда',
 'появился',
 'кот',
 'сначала',
 'его',
 'заметил',
 'маленький',
 'котик']

In [ ]:
len(words)

49

In [ ]:
def add_postfix(postfix, cur_level, end_symbol='$'):
    """
    Функция добавляет postfix в поддерево с узлом cur_level.
    Узел дерева хранит в нулевом элементе частоту встречаемости пройденного префикса.
    В первом элементе хранится словарь всех элементов, с которых могут
    начинаться возможные суффиксы, начинающиеся с данного префикса.
    """

    # Префикс закончился (его постфикс пуст). Добавлять больше нечего.
    if len(postfix) == 0:
        if end_symbol not in cur_level:
            cur_level[end_symbol] = [0, {}]
        cur_level[end_symbol][0] += 1
        return

    # Такого начала нет в словаре. Его надо добавить.
    if postfix[0] not in cur_level:
        cur_level[postfix[0]] = [0, {}]

    # Пришел еще один такой префикс, увеличиваем частоту.
    cur_level[postfix[0]][0] += 1

    # Добавляем оставшуюся часть в соответствующий узел.
    add_postfix(postfix[1:], cur_level[postfix[0]][1])

In [ ]:
def print_tree(node, prefix="", end_symbol='$'):
    """
    Печатает дерево в виде ASCII-графа.
    Узлы выводятся в алфавитном порядке.
    end_symbol ('$') трактуется как конец слова.
    """

    # Сортируем ключи, чтобы вывод был по алфавиту
    for i, ch in enumerate(sorted(node.keys())):

        cnt, children = node[ch]

        # Проверяем, последний ли это элемент на текущем уровне
        is_last = (i == len(node) - 1)

        # Выбираем соединительный символ
        connector = "└── " if is_last else "├── "

        # Отдельно красиво отображаем конец слова
        if ch == end_symbol:
            print(f"{prefix}{connector}{end_symbol} ({cnt})")
        else:
            print(f"{prefix}{connector}{ch} ({cnt})")

        # Выбираем продолжение отступа
        extension = "    " if is_last else "│   "

        # Рекурсивно печатаем поддерево
        print_tree(children, prefix + extension)

In [ ]:
trie = {}
for word in words:
    add_postfix(word, trie)
print_tree(trie)

├── а (1)
│   └── $ (1)
├── б (1)
│   └── л (1)
│       └── и (1)
│           └── ж (1)
│               └── е (1)
│                   └── $ (1)
├── в (5)
│   ├── $ (1)
│   ├── з (1)
│   │   └── р (1)
│   │       └── о (1)
│   │           └── с (1)
│   │               └── л (1)
│   │                   └── ы (1)
│   │                       └── х (1)
│   │                           └── $ (1)
│   ├── о (1)
│   │   └── к (1)
│   │       └── р (1)
│   │           └── у (1)
│   │               └── г (1)
│   │                   └── $ (1)
│   └── с (2)
│       ├── е (1)
│       │   └── $ (1)
│       └── к (1)
│           └── о (1)
│               └── р (1)
│                   └── е (1)
│                       └── $ (1)
├── д (3)
│   ├── в (1)
│   │   └── о (1)
│   │       └── р (1)
│   │           └── е (1)
│   │               └── $ (1)
│   ├── о (1)
│   │   └── м (1)
│   │       └── $ (1)
│   └── р (1)
│       └── у (1)
│           └── г (1)
│               └── и (1)
│                   └── е 

In [ ]:
def find_prefix(prefix, cur_level):
    """
    Ищет prefix в суффиксном поддереве cur_level.
    """

    # Префикс пустой. Ошибка.
    if len(prefix) == 0:
        return -2

    # Очередного элемента нет в узле префиксного дерева. Ошибка.
    if cur_level.get(prefix[0], None) is None:
        return -1

    # Элемент есть в узле, но он последний в префиксе. Возвращаем результат.
    if len(prefix) == 1:
        return cur_level[prefix[0]][0]

    # Поиск не закончен, переходим на следующий уровень
    return find_prefix(prefix[1:], cur_level[prefix[0]][1])

In [ ]:
find_prefix('кот', trie)

10

In [ ]:
find_prefix('кот$', trie)

2

In [ ]:
find_prefix('котовасия', trie)

-1

Суффиксное дерево отличается от префиксного только тем, что слова помещаются в него с конца. Это, например, может быть актуально для русского с преимущественно суффиксальной морфологией.

## Двоичное дерево

Еще одним вариантом хранения данных является двоичное дерево - структура, где слева от текущей вершины находятся все элементы меньше нее, а справа - больше. Причем у каждой вершины может быть только два потомка - правы и левый.

<img src="https://github.com/klyshinsky/Math4Linguists2025/blob/main/img/7f3d0251cd5f42e8a0a1f3aec41da199.png?raw=1" width="300px">

Как искать в таким дереве?

1. Начиная с корня применяем шаги 2-4.
2. Если искомое значение равно значению в вершине, то значение найдено. Успешно завершаем работу алгоритма.
3. Если искомое значение меньше, чем значение в вершине, переходим к левому потомку. Если левый потомок отсутствует, завершаем алгоритм неуспехом.
4. Если искомое значение больше, чем значение в вершине, переходим к правому потомку. Если правый потомок отсутствует, завершаем алгоритм неуспехом.

Скорость поиска по такому дереву - примерно двоичный логарифм от количества вершин в дереве.

In [ ]:
# Набор функций для создания двоичного дерева.

def add_value_to_node(node, value):
    """
    Функция добавления значения к вершине.
    """
    # Такое значение уже есть в в дереве.
    if node[0] == value:
        return
    # Значение должно пойти налево.
    if value < node[0]:
        # Слева никого нет.
        if node[1] is None:
            # [текущее значение, левый потомок, правый потомок]
            node[1] = [value, None, None]
        # Левый потомок имеется, рекурсивно применяем алгоритм.
        else:
            add_value_to_node(node[1], value)
    # Аналогично справа.
    else:
        if node[2] is None:
            node[2] = [value, None, None]
        else:
            add_value_to_node(node[2], value)


def add_value_to_tree(root, value):
    """
    Функция добавления значения к дереву.
    """
    # Если корень дерева не создан, то надо его создать.
    if root is None or root == []:
        return [value, None, None]
    # Если корень существует, то просим его добавить начение.
    add_value_to_node(root, value)
    return root

In [ ]:
def print_bst(node, prefix="", branch=""):
    """
    Красивый вывод BST с подписями веток (< и >).
    Формат узла: [value, left, right]
    """

    if node is None:
        return

    value, left, right = node

    # Корневой узел
    if prefix == "":
        print(value)
    else:
        print(f"{prefix}{branch}{value}")

    # Новый префикс для детей
    new_prefix = prefix + ("    " if branch == "< " else "│   ")

    # Сначала правое поддерево (>)
    if right is not None:
        print_bst(right, new_prefix, "> ")

    # Потом левое поддерево (<)
    if left is not None:
        print_bst(left, new_prefix, "< ")

Посмотрим как работает.

In [ ]:
tree = None

In [ ]:
tree = add_value_to_tree(tree, 5)
print_bst(tree)

5


In [ ]:
tree = add_value_to_tree(tree, 5)
tree = add_value_to_tree(tree, 4)
tree = add_value_to_tree(tree, 3)
print_bst(tree)

5
│   < 4
│       < 3


In [ ]:
tree = add_value_to_tree(tree, 9)
tree = add_value_to_tree(tree, 12)
tree = add_value_to_tree(tree, 7)
print_bst(tree)

5
│   > 9
│   │   > 12
│   │   < 7
│   < 4
│       < 3


Если говорить о построении такого дерева по списку элементов, то достаточно логично сначала отсортировать список, а потом разложить его в дерево следующим способом:
1. Берём элемент на позиции N//2 и делаем его корнем текущего поддерева.
2. Все элементы левее него образуют левое поддерево, все элементы правее — правое поддерево.
3. Рекурсивно применяем те же шаги к левому и правому поддеревьям, пока подмассивы не станут пустыми.

In [ ]:
def build_bst_from_sorted(lst):
    """
    Строит сбалансированное BST из отсортированного списка.
    Формат узла: [value, left, right]
    """

    if not lst:
        return None

    mid = len(lst) // 2

    root = [lst[mid], None, None]

    # Рекурсивно строим левое и правое поддерево
    root[1] = build_bst_from_sorted(lst[:mid])
    root[2] = build_bst_from_sorted(lst[mid + 1:])

    return root

In [ ]:
# отсортированный список
lst = list(range(10))

tree = build_bst_from_sorted(lst)
print_bst(tree)

5
│   > 8
│   │   > 9
│   │   < 7
│   │       < 6
│   < 2
│       > 4
│       │   < 3
│       < 1
│           < 0


Визуально дерево получилось сбалансированным. Однако, если мы работаем в режиме, когда к нам приходят новые элементы и мы последовательно добавляем их в дерево, то отсортированный массив создаст нам большие проблемы:

In [ ]:
tree = []
for i in range(10):
    tree = add_value_to_tree(tree, i)
print_bst(tree)

0
│   > 1
│   │   > 2
│   │   │   > 3
│   │   │   │   > 4
│   │   │   │   │   > 5
│   │   │   │   │   │   > 6
│   │   │   │   │   │   │   > 7
│   │   │   │   │   │   │   │   > 8
│   │   │   │   │   │   │   │   │   > 9


Очевидно, что скорость поиска по такому дереву мало будет отличаться от поиска по неотсортированному массиву.



### АВЛ-деревья и их балансировка

Необходимо поддерживать *сбалансированность* дерева. Для каждой вершины сбалансированного дерева высота её двух поддеревьев различается не более чем на 1.

Здесь мы рассмотрим [АВЛ-деревья](https://ru.wikipedia.org/wiki/%D0%90%D0%92%D0%9B-%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D0%BE).

Балансировка АВЛ дерева заключается в следующем. Балансировкой вершины называется операция, которая в случае разницы высот левого и правого поддеревьев = 2, изменяет связи предок-потомок в поддереве данной вершины так, что разница становится <= 1, иначе ничего не меняет. Указанный результат получается вращениями поддерева данной вершины.

Используются 4 типа вращений:
- Малое левое вращение  

<img src="https://upload.wikimedia.org/wikipedia/commons/9/96/AVL_LR.gif">  

Данное вращение используется тогда, когда (высота b-поддерева — высота L) = 2 и высота c-поддерева <= высота R.

- Большое левое вращение  

<img src="https://upload.wikimedia.org/wikipedia/ru/1/16/AVL_BR.GIF">  

Данное вращение используется тогда, когда (высота b-поддерева — высота L) = 2 и высота c-поддерева > высота R.

- Малое правое вращение  

<img src="https://upload.wikimedia.org/wikipedia/ru/e/e8/AVL_LL.GIF">  

Данное вращение используется тогда, когда (высота b-поддерева — высота R) = 2 и высота С <= высота L.

- Большое правое вращение  

<img src="https://upload.wikimedia.org/wikipedia/ru/7/74/AVL_BL.GIF">  

Данное вращение используется тогда, когда (высота b-поддерева — высота R) = 2 и высота c-поддерева > высота L.

Аналогичный результат можно достигнуть при помощи [красно-черных деревьев](https://neerc.ifmo.ru/wiki/index.php?title=%D0%9A%D1%80%D0%B0%D1%81%D0%BD%D0%BE-%D1%87%D0%B5%D1%80%D0%BD%D0%BE%D0%B5_%D0%B4%D0%B5%D1%80%D0%B5%D0%B2%D0%BE), которые мы не будем рассматривать в рамках данного курса. Их общая идея балансировки также основывается на поворотах, но когда применять повороты определяется посредством окраски вершин дерева в красные и черные цвета. За счет этого достигается некоторая экономия на подсчетах глубины дерева.

## B-деревья

Бинарные деревья являются частным случаем [В-деревьев](https://habr.com/ru/post/114154/). (Поиграть с ними можно [здесь](https://www.cs.usfca.edu/~galles/visualization/BTree.html))

В В-дереве каждая вершина хранит от N-1 до 2*N-1 элемента данных (ключа) в отсортированном виде. Число потомков - на одного больше. Все листья В-дерева должны быть располоены на одной высоте (на практике подобное требование может быть ослаблено до разницы уровней не больше 1 для любой вершины).

<img src="https://hsto.org/getpro/habr/post_images/1f2/f68/57e/1f2f6857e2fa99d307f1b6b371a7fc0e.png">

Поиск осуществляется следущим образом.
1. Назначаем корень дерева текущей вершиной.
2. Идем по ключам текущей вершины до величины, большей искомой. Если такая найдена, переходим к потомку, который находится перед этой вершиной. Если не найдена - переходим к последнему потомку.
3. Повторяем шаг 2 до тех пор, пока не будет найден элемент или пока есть потомки.

##### Добавление элементов
Если лист был заполнен, то в нем находилось 2N-1 ключей, то есть его можно разбить на 2 по N-1. При этом средний элемент (для которого N-1 первых ключей меньше его, а N-1 последних больше) перемещается в родительский узел. Соответственно, если родительский узел также был заполнен – то нам опять приходится разбивать. Операция может повторяться до корня. Если разбивается корень – то появляется новый корень и глубина дерева увеличивается. Как и в случае обычных бинарных деревьев, вставка осуществляется за один проход от корня к листу. На каждой итерации мы разбиваем все заполненные узлы, через которые проходим.

Вместо того, чтобы разбивать узел, мы можем попробовать передать один из его элементов одному из потомков (например, самый левый элемент левому потомку). Очевидно, что при этом выбирают того из потомков, который не заполнен до конца.

<img src="https://hsto.org/getpro/habr/post_images/21b/18f/afb/21b18fafb0065fba63b60477c4d820fb.png">

<img src="https://hsto.org/getpro/habr/post_images/00f/28b/100/00f28b100bfb1cefb0f2df9a2208f337.png">

##### Удаление элементов
Если удаление происходит из листа, то необходимо проверить, сколько ключей в нем находится. Если больше N-1, то просто удаляем. Иначе, если существует соседний лист, который содержит больше N-1 ключа, то выберем ключ из этого соседа, который является разделителем между оставшимися ключами узла-соседа и исходного узла, и перенем в текущую вершину, а выбранное значение из узла-соседа перенесем в родительский узел. Если все узлы имеют по N-1 ключу, то можно объединить двух потомков, удалив из них нужный ключ.

<img src="https://hsto.org/getpro/habr/post_images/a50/1c9/f67/a501c9f67b746de70ca2ad11b5b0a085.png">

Если удаление производится из внутренней вершины, то можно взять себе ключ из одного из потомков. Если дочерний узел, предшествующий удаляемому ключу k содержит больше N-1 ключа, то находим k1 – предшественника k в поддереве этого узла. Удаляем его (возможно, рекурсивно запуская удаление дальше) и заменяем k в исходном узле на k1.

<img src="https://hsto.org/getpro/habr/post_images/3de/f86/2c4/3def862c44eb090f9684a8fdbcf1f516.png">

##### Зачем они нужны
В-деревья хорошо показывют себя в системах, в которых переход с уровня на уровень стоит дорого. Например, при хранении информации на диске или в базе данных. Применение В-деревьев позволяет сократить количество обращений к носителю, то есть ускорить поиск данных.

## Интересная идея

**Реализация Т9 на префиксном дереве.**

Нам надо реализовать Т9 (подсказки пользователю, набирающему текст на клавиатуре). Суть Т9 заключается в том, что пользователю выдаются слова, наиболее частные в каком-то тексте (например, во всех текстах, набранных данным пользователем). Тогда с одной стороны, нам нужно хранить частоты слов, а с другой стороны, флаг окончания слова.

Алгоритм составления дерева для Т9 будет следующим. Добавляем все слова из некотрого текста. Для каждой буквы, которую мы проходим, увеличиваем частоту ее встречаемости. В конце слова устанавливаем флаг окончания слова в истинное значение (по умолчанию - ложное).

Перед началом поиска слова делаем текущей корневую вершину. При вводе очередной буквы переходим в вершину, соответствующей данной буквы. После этого, выбираем ветвь дерева с наибольшей частотой. Повторяем операцию до первой вершины с установленным флагом конца слова. Полученный путь в дереве выдаем в качестве подсказки.
